In [ ]:
!pip install numpy pandas matplotlib yfinance tensorflow scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime, timedelta
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
import warnings
warnings.filterwarnings('ignore')

# Function to get stock data
def get_stock_data(ticker, period='5y'):
    """
    Fetch stock data from Yahoo Finance

    Parameters:
    ticker (str): Stock ticker symbol
    period (str): Period of historical data to fetch

    Returns:
    DataFrame: Stock data
    """
    try:
        stock = yf.Ticker(ticker)
        df = stock.history(period=period)
        if len(df) == 0:
            raise ValueError(f"No data found for ticker {ticker}")
        return df
    except Exception as e:
        print(f"Error fetching data: {e}")
        return None

# Function to prepare data for LSTM
def prepare_data(df, feature='Close', look_back=60):
    """
    Prepare data for LSTM model

    Parameters:
    df (DataFrame): Stock data
    feature (str): Feature to predict
    look_back (int): Number of previous days to use for prediction

    Returns:
    tuple: X_train, y_train, X_test, y_test, scaler
    """
    # Select only the relevant feature
    data = df[feature].values.reshape(-1, 1)

    # Scale the data
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(data)

    # Split data into training (80%) and testing (20%) sets
    train_size = int(len(scaled_data) * 0.8)
    train_data = scaled_data[:train_size]
    test_data = scaled_data[train_size - look_back:]

    # Create sequences for training
    X_train, y_train = [], []
    for i in range(look_back, len(train_data)):
        X_train.append(train_data[i - look_back:i, 0])
        y_train.append(train_data[i, 0])

    # Create sequences for testing
    X_test, y_test = [], []
    for i in range(look_back, len(test_data)):
        X_test.append(test_data[i - look_back:i, 0])
        y_test.append(test_data[i, 0])

    # Convert to numpy arrays
    X_train, y_train = np.array(X_train), np.array(y_train)
    X_test, y_test = np.array(X_test), np.array(y_test)

    # Reshape for LSTM [samples, time steps, features]
    X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
    X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

    return X_train, y_train, X_test, y_test, scaler, train_size, scaled_data

# Function to build LSTM model
def build_model(look_back):
    """
    Build LSTM model

    Parameters:
    look_back (int): Number of previous time steps to use

    Returns:
    Sequential: LSTM model
    """
    model = Sequential([
        LSTM(units=50, return_sequences=True, input_shape=(look_back, 1)),
        Dropout(0.2),
        LSTM(units=50, return_sequences=False),
        Dropout(0.2),
        Dense(units=25),
        Dense(units=1)
    ])

    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

# Function to make future predictions
def predict_future(model, last_sequence, scaler, days=30):
    """
    Make future predictions

    Parameters:
    model: Trained LSTM model
    last_sequence: Last sequence of data
    scaler: Fitted scaler for inverse transformation
    days (int): Number of days to predict

    Returns:
    array: Future predictions
    """
    future_predictions = []
    current_sequence = last_sequence.copy()

    for _ in range(days):
        # Reshape for model input
        current_sequence_reshaped = np.reshape(current_sequence, (1, current_sequence.shape[0], 1))

        # Get prediction (next day)
        next_day_prediction = model.predict(current_sequence_reshaped, verbose=0)[0, 0]

        # Add prediction to results
        future_predictions.append(next_day_prediction)

        # Update sequence (remove first element, append prediction)
        current_sequence = np.append(current_sequence[1:], next_day_prediction)

    # Inverse transform to get actual values
    future_predictions = np.array(future_predictions).reshape(-1, 1)
    future_predictions = scaler.inverse_transform(future_predictions)

    return future_predictions

# Function to evaluate model
def evaluate_model(y_test, predictions, scaler):
    """
    Evaluate model performance

    Parameters:
    y_test: Actual test values
    predictions: Predicted values
    scaler: Fitted scaler for inverse transformation

    Returns:
    tuple: RMSE, MAPE
    """
    # Convert predictions to original scale
    y_test_orig = scaler.inverse_transform(np.array(y_test).reshape(-1, 1))
    predictions_orig = scaler.inverse_transform(predictions.reshape(-1, 1))

    # Calculate RMSE
    rmse = np.sqrt(np.mean((predictions_orig - y_test_orig) ** 2))

    # Calculate MAPE
    mape = np.mean(np.abs((y_test_orig - predictions_orig) / y_test_orig)) * 100

    return rmse, mape

# Function to visualize results
def visualize_results(df, feature, predictions, future_predictions, train_size, look_back, ticker):
    """
    Visualize historical prices and predictions

    Parameters:
    df: Stock data
    feature: Feature being predicted
    predictions: Test set predictions
    future_predictions: Future predictions
    train_size: Size of training data
    look_back: Number of previous time steps used
    ticker: Stock ticker symbol
    """
    # Create a figure with subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 12))

    # Plot 1: Historical data and test predictions
    ax1.set_title(f'{ticker} Stock Price Prediction', fontsize=16)
    ax1.plot(df.index, df[feature], label='Historical Prices', color='blue')

    # Get the dates for the test predictions - only include non-NaN values
    valid_indices = ~np.isnan(predictions.flatten())
    test_prediction_dates = df.index[valid_indices]
    valid_predictions = predictions[valid_indices].flatten()

    # Make sure test_prediction_dates and valid_predictions have the same length
    if len(test_prediction_dates) > len(valid_predictions):
        test_prediction_dates = test_prediction_dates[-len(valid_predictions):]
    elif len(valid_predictions) > len(test_prediction_dates):
        valid_predictions = valid_predictions[-len(test_prediction_dates):]

    ax1.plot(test_prediction_dates, valid_predictions, label='Test Predictions', color='red')

    # Add a vertical line to separate training and testing data
    ax1.axvline(x=df.index[train_size], color='green', linestyle='--', label='Train-Test Split')

    ax1.set_xlabel('Date', fontsize=12)
    ax1.set_ylabel(f'{feature} Price ($)', fontsize=12)
    ax1.legend()
    ax1.grid(True)

    # Plot 2: Future predictions
    last_date = df.index[-1]
    future_dates = [last_date + timedelta(days=i) for i in range(1, len(future_predictions) + 1)]

    # Plot the last 90 days of historical data for context
    historical_subset = df.iloc[-90:][feature]
    ax2.plot(historical_subset.index, historical_subset, label='Recent Historical Prices', color='blue')

    # Plot future predictions
    future_prices = future_predictions.flatten()
    ax2.plot(future_dates, future_prices, label='Future Predictions', color='red')

    # Add a vertical line to separate historical and future data
    ax2.axvline(x=last_date, color='green', linestyle='--', label='Prediction Start Point')

    ax2.set_title(f'{ticker} Future Price Prediction (Next {len(future_predictions)} Days)', fontsize=16)
    ax2.set_xlabel('Date', fontsize=12)
    ax2.set_ylabel(f'{feature} Price ($)', fontsize=12)
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

# Main function
def main():
    # Get user input for stock ticker
    ticker = input("Enter stock ticker symbol (e.g., AAPL, MSFT, GOOGL): ").strip().upper()

    print(f"Fetching historical data for {ticker}...")
    df = get_stock_data(ticker)

    if df is None or len(df) < 100:
        print(f"Insufficient data for {ticker}. Please try another ticker.")
        return

    print(f"Data fetched successfully! Historical data from {df.index[0].date()} to {df.index[-1].date()}")

    # Data preparation
    look_back = 60  # Using 60 days of historical data for prediction
    feature = 'Close'  # Predicting closing prices

    X_train, y_train, X_test, y_test, scaler, train_size, scaled_data = prepare_data(df, feature, look_back)

    print("Building and training LSTM model...")

    # Build and train the model
    model = build_model(look_back)
    history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.1, verbose=1)

    # Make predictions on test data
    test_predictions = model.predict(X_test)

    # Evaluate model
    rmse, mape = evaluate_model(y_test, test_predictions, scaler)
    print(f"Model Evaluation Metrics:")
    print(f"RMSE: ${rmse:.2f}")
    print(f"MAPE: {mape:.2f}%")

    # Predict future prices (30 days)
    print("Predicting future stock prices...")
    last_sequence = scaled_data[-look_back:]
    future_days = 30
    future_predictions = predict_future(model, last_sequence, scaler, future_days)

    # Get the last date in our dataset to forecast future dates
    last_date = df.index[-1]
    future_dates = [last_date + timedelta(days=i) for i in range(1, future_days + 1)]

    print(f"\nFuture price predictions for {ticker} (next {future_days} days):")
    for i, (date, price) in enumerate(zip(future_dates, future_predictions)):
        if i % 5 == 0 or i == len(future_dates) - 1:  # Print every 5 days to keep output manageable
            print(f"{date.date()}: ${price[0]:.2f}")

    # Prepare predictions for visualization
    # Create an array of NaN with the same length as the original dataframe
    test_predictions_full = np.zeros((len(df), 1))
    test_predictions_full[:] = np.nan

    # Place the test predictions in the correct positions
    # Make sure we don't exceed array bounds
    start_idx = train_size
    end_idx = min(start_idx + len(test_predictions), len(test_predictions_full))
    pred_length = end_idx - start_idx

    # Only use as many predictions as we have space for
    test_predictions_to_use = test_predictions[:pred_length]
    test_predictions_full[start_idx:end_idx] = scaler.inverse_transform(test_predictions_to_use)

    # Visualize results
    visualize_results(df, feature, test_predictions_full, future_predictions, train_size, look_back, ticker)

if __name__ == "__main__":
    main()